# Entrenamiento Personalizado: StyleGan2-ADA

En este cuaderno, realizaremos transfer learning con StyleGAN2 y conjuntos de datos personalizados.

Esto significa que no entrenaremos la red GAN desde cero con nuestras imágenes (ya que toma alrededor de dos semanas), sino que usaremos el modelo ya entrenado en otras imágenes como punto de partida. Esto reducirá el tiempo de entrenamiento a unas horas al omitir las etapas iniciales donde la red neuronal aprende características de bajo nivel de imágenes que son muy similares para cualquier tipo de imágenes.

In [1]:
#@title 1. Conectar Colab a  Google Drive
#@markdown Accede a tu Google Drive para cargar tu conjunto de datos, editar las imágenes y guardar los resultados.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#@title 2. Instalación de StyleGAN2-ADA
#@markdown Esta celda prepara StyleGAN2-ADA para la versión actual de Google Colab.
#@markdown
#@markdown El código se instala temporalmente en Colab para obtener mejor rendimiento.
#@markdown Tus datasets, modelos y resultados se guardarán en Google Drive y no se perderán al cerrar la sesión.
#@markdown
#@markdown Ejecuta esta celda una vez al comenzar cada nueva sesión de Colab.

import os
import sys
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

# ------------------------------------------------------------
# 1. Conectar Google Drive
# ------------------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Crear carpetas persistentes en Google Drive
# ------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/colab-sg2-ada-pytorch")

DATASETS_DIR   = DRIVE_ROOT / "datasets"
RESULTS_DIR    = DRIVE_ROOT / "results"
PRETRAINED_DIR = DRIVE_ROOT / "pretrained"

for folder in [DATASETS_DIR, RESULTS_DIR, PRETRAINED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Instalar dependencias necesarias
# ------------------------------------------------------------

print("Instalando dependencias...")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "ninja",
        "imageio-ffmpeg",
        "click",
        "requests",
        "tqdm"
    ],
    check=True
)

# ------------------------------------------------------------
# 4. Descargar StyleGAN2-ADA
# ------------------------------------------------------------

REPO_DIR = Path("/content/stylegan2-ada-pytorch")
REPO_URL = "https://github.com/angelv-salazar/stylegan2-ada-pytorch.git"
BRANCH = "colab-2026"

# El repositorio dentro de /content es temporal.
# Lo descargamos limpio al iniciar cada sesión.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print("Descargando StyleGAN2-ADA...")

subprocess.run(
    [
        "git", "clone",
        "--branch", BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR)
    ],
    check=True
)

os.chdir(REPO_DIR)

# ------------------------------------------------------------
# 5. Verificación
# ------------------------------------------------------------

import torch

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    text=True
).strip()

print("\n----------------------------------------")
print("StyleGAN2-ADA instalado correctamente")
print("----------------------------------------")
print("Repositorio:", REPO_DIR)
print("Rama:", branch)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("\n✅ Todo listo para entrenar.")
else:
    print("\n⚠️ No se detectó una GPU.")
    print("Activa una GPU desde Entorno de ejecución > Cambiar tipo de entorno de ejecución.")

In [ ]:
#@title 3. Preparar Dataset: Recorte, Redimensionamiento y Compresión ZIP
#@markdown Permite recortar y preparar imágenes cuadradas para StyleGAN2, y luego comprimir la carpeta final en un ZIP listo para el entrenamiento.

import os, glob, zipfile, traceback
from math import ceil, floor
from PIL import Image
import numpy as np
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------
# 🔹 Funciones auxiliares
# ---------------------------
def crop_center(img, ratio=1.0):
    w, h = img.size
    min_side = min(w, h)
    crop_size = int(min_side * ratio)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size
    return img.crop((left, top, right, bottom))

def save_crop(img, count, out_dir, augment=False):
    img.save(f'{out_dir}{count:08d}.png')
    count += 1
    if augment:
        for angle in [90, 180, 270]:
            img_rot = img.rotate(angle, expand=True)
            img_rot.save(f'{out_dir}{count:08d}.png')
            count += 1
    return count

# ---------------------------
# 🔹 Widgets de interfaz
# ---------------------------
input_dir_widget = widgets.Text(
    value="/content/drive/MyDrive/amonites",
    description="Origen:",
    layout=widgets.Layout(width='95%')
)

output_dir_widget = widgets.Text(
    value="/content/drive/MyDrive/colab-sg2-ada-pytorch/datasets/amonites",
    description="Salida:",
    layout=widgets.Layout(width='95%')
)

resize_widget = widgets.Dropdown(
    options=[256, 512, 1024],
    value=512,
    description="Tamaño:",
    layout=widgets.Layout(width='45%')
)

ratio_widget = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=1.0,
    step=0.05,
    description="🔳 Ratio:",
    continuous_update=True,
    layout=widgets.Layout(width='45%')
)

augment_widget = widgets.Checkbox(
    value=False,
    description="Aumentar dataset con rotaciones",
    layout=widgets.Layout(width='60%')
)

process_button = widgets.Button(
    description="🚀 Procesar imágenes",
    button_style='success',
    layout=widgets.Layout(width='60%', margin='10px 0px 10px 0px')
)

zip_button = widgets.Button(
    description="📦 Comprimir carpeta en ZIP",
    button_style='info',
    layout=widgets.Layout(width='60%', margin='10px 0px 10px 0px')
)

progress_label = widgets.Label(value="")
progress_bar = widgets.FloatProgress(value=0.0, min=0.0, max=1.0, layout=widgets.Layout(width='90%'))
error_output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px'))
output_area = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px'))

# ---------------------------
# 🔹 Función principal de procesamiento
# ---------------------------
def process_dataset(b):
    clear_output(wait=True)
    display(ui)

    input_dir = input_dir_widget.value.strip()
    out_dir = output_dir_widget.value.strip()
    resize = int(resize_widget.value)
    ratio = ratio_widget.value
    augment = augment_widget.value

    if not out_dir.endswith("/"):
        out_dir += "/"

    # ❌ Verificar si hay espacios en la ruta
    if " " in out_dir or " " in input_dir:
        with output_area:
            clear_output(wait=True)
            print("⚠️ Error: No se permiten espacios en la ruta de origen o salida. Renombra las carpetas y vuelve a intentar.")
        return

    os.makedirs(out_dir, exist_ok=True)

    images = glob.glob(f'{input_dir}/*.tif') + \
             glob.glob(f'{input_dir}/*.png') + \
             glob.glob(f'{input_dir}/*.jpg') + \
             glob.glob(f'{input_dir}/*.jpeg') + \
             glob.glob(f'{input_dir}/*.bmp')

    total_images = len(images)
    cnt = 0
    failed = []

    with output_area:
        clear_output(wait=True)
        if total_images == 0:
            print("⚠️ No se encontraron imágenes en la carpeta de origen.")
            return
        print(f'📂 Procesando {total_images} imágenes...\n')

    progress_bar.value = 0
    progress_label.value = f"0 / {total_images} procesadas"

    for i, img_path in enumerate(images):
        try:
            img = Image.open(img_path)
            img = crop_center(img, ratio)
            img = img.resize((resize, resize), Image.Resampling.LANCZOS)

            if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
                bg = Image.new('RGB', img.size, (255, 255, 255))
                bg.paste(img, (0, 0), img)
                img = bg
            elif img.mode != 'RGB':
                img = img.convert('RGB')

            cnt = save_crop(img, cnt, out_dir, augment)
        except Exception as e:
            failed.append((os.path.basename(img_path), str(e)))

        progress_bar.value = (i + 1) / total_images
        progress_label.value = f"{i+1} / {total_images} procesadas ({(i+1)/total_images*100:.1f}%)"

    with output_area:
        print(f'\n✅ Procesamiento completado.')
        print(f'🖼️ Total de imágenes guardadas: {cnt}')
        print(f'📁 Directorio de salida: {out_dir}')

    with error_output:
        clear_output(wait=True)
        if failed:
            print(f"\n⚠️ {len(failed)} imágenes no pudieron procesarse:\n")
            for name, err in failed:
                print(f"   • {name} → {err}")
        else:
            print("✅ Todas las imágenes fueron procesadas correctamente.")

# ---------------------------
# 🔹 Función de compresión ZIP
# ---------------------------
def compress_dataset(b):
    clear_output(wait=True)
    display(ui)

    out_dir = output_dir_widget.value.strip()
    if not out_dir.endswith("/"):
        out_dir += "/"

    if " " in out_dir:
        with output_area:
            clear_output(wait=True)
            print("⚠️ Error: No se permiten espacios en la ruta del dataset. Renombra la carpeta y vuelve a intentar.")
        return

    if not os.path.exists(out_dir):
        with output_area:
            clear_output(wait=True)
            print("⚠️ La carpeta de salida no existe. Procesa las imágenes primero.")
        return

    zip_path = out_dir.rstrip("/") + ".zip"
    files = glob.glob(os.path.join(out_dir, "*.*"))
    total_files = len(files)

    if total_files == 0:
        with output_area:
            clear_output(wait=True)
            print("⚠️ No hay imágenes para comprimir.")
        return

    with output_area:
        clear_output(wait=True)
        print(f"📦 Comprimiendo {total_files} archivos en: {zip_path}\n")

    progress_bar.value = 0
    progress_label.value = "0% completado"

    try:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
            for i, file in enumerate(files):
                arcname = os.path.basename(file)
                zipf.write(file, arcname)
                progress_bar.value = (i + 1) / total_files
                progress_label.value = f"{(i + 1) / total_files * 100:.1f}% completado"

        with output_area:
            print(f"\n✅ Carpeta comprimida correctamente.")
            print(f"📁 Archivo ZIP creado: {zip_path}")
    except Exception as e:
        with error_output:
            clear_output(wait=True)
            print(f"⚠️ Error al comprimir: {str(e)}")

# ---------------------------
# 🔹 Eventos de botones
# ---------------------------
process_button.on_click(process_dataset)
zip_button.on_click(compress_dataset)

# ---------------------------
# 🔹 Interfaz principal
# ---------------------------
ui = widgets.VBox([
    widgets.HTML("<h3>🧩 Preparar Dataset: Recorte, Redimensionamiento y Compresión ZIP</h3>"),
    input_dir_widget,
    output_dir_widget,
    widgets.HBox([resize_widget, ratio_widget]),
    augment_widget,
    process_button,
    zip_button,
    progress_bar,
    progress_label,
    output_area,
    error_output
])

display(ui)


In [ ]:
#@title 4. Entrenamiento de Modelo StyleGAN2 Personalizado

#@markdown Selecciona el archivo `.zip` de tu dataset.
#@markdown
#@markdown El nombre del archivo será utilizado automáticamente como identificador
#@markdown del entrenamiento. Por ejemplo, `hongos.zip` guardará sus resultados
#@markdown dentro de una carpeta llamada `hongos`.

dataset = "/content/drive/MyDrive/colab-sg2-ada-pytorch/datasets/amonites.zip" #@param {type:"string"}

#@markdown ---
#@markdown ### Inicio / reanudación
#@markdown Si está activado, el sistema buscará automáticamente el último `.pkl`
#@markdown correspondiente a este dataset y continuará desde allí.
#@markdown
#@markdown Si no encuentra ningún entrenamiento anterior, comenzará desde el
#@markdown modelo base seleccionado abajo.

reanudar_automaticamente = True #@param {type:"boolean"}

modelo_inicial = "ffhq512" #@param ["ffhq512", "ffhq1024"]

#@markdown ---
#@markdown ### Aumentos por reflexión
#@markdown Activa estas opciones únicamente si reflejar las imágenes tiene sentido
#@markdown para tu dataset.

mirror_horizontal = False #@param {type:"boolean"}
mirror_vertical = False #@param {type:"boolean"}


# ============================================================
# NO ES NECESARIO EDITAR DEBAJO DE ESTA LÍNEA
# ============================================================

from pathlib import Path
import os
import re
import subprocess
import legacy


# ------------------------------------------------------------
# Directorio de StyleGAN2-ADA
# ------------------------------------------------------------

REPO_DIR = Path("/content/stylegan2-ada-pytorch")

if not REPO_DIR.exists():
    raise FileNotFoundError(
        "No se encontró StyleGAN2-ADA.\n"
        "Ejecuta primero la celda de instalación."
    )

os.chdir(REPO_DIR)


# ------------------------------------------------------------
# Comprobar dataset
# ------------------------------------------------------------

DATASET = Path(dataset)

if not DATASET.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset:\n{DATASET}"
    )


# ------------------------------------------------------------
# Identificador del proyecto
# ------------------------------------------------------------

PROJECT_NAME = DATASET.stem

# Limpiar caracteres problemáticos
PROJECT_NAME = re.sub(
    r"[^a-zA-Z0-9._-]+",
    "_",
    PROJECT_NAME
)

RESULTS_ROOT = Path(
    "/content/drive/MyDrive/colab-sg2-ada-pytorch/results"
)

PROJECT_DIR = RESULTS_ROOT / PROJECT_NAME
PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("========================================")
print("STYLEGAN2-ADA")
print("========================================")
print("Dataset:   ", DATASET)
print("Proyecto:  ", PROJECT_NAME)
print("Resultados:", PROJECT_DIR)


# ------------------------------------------------------------
# Función para identificar los kimg de cada snapshot
# ------------------------------------------------------------

def get_snapshot_kimg(path):

    match = re.search(
        r"network-snapshot-(\d+)\.pkl$",
        path.name
    )

    if match:
        return int(match.group(1))

    return -1


# ------------------------------------------------------------
# Buscar snapshots anteriores de ESTE dataset
# ------------------------------------------------------------

snapshots = list(
    PROJECT_DIR.glob(
        "**/network-snapshot-*.pkl"
    )
)

START_KIMG = 0
ADA_P = 0.0


# ------------------------------------------------------------
# Reanudar entrenamiento
# ------------------------------------------------------------

if reanudar_automaticamente and snapshots:

    latest_snapshot = max(
        snapshots,
        key=get_snapshot_kimg
    )

    START_KIMG = get_snapshot_kimg(
        latest_snapshot
    )

    RESUME = str(
        latest_snapshot
    )

    print()
    print("Entrenamiento anterior encontrado.")
    print("Snapshot:", latest_snapshot)
    print(
        "Continuando desde:",
        START_KIMG,
        "kimg"
    )

    # Recuperar intensidad ADA
    try:

        with open(
            latest_snapshot,
            "rb"
        ) as f:

            snapshot_data = legacy.load_network_pkl(
                f
            )

        augment_pipe = snapshot_data.get(
            "augment_pipe"
        )

        if augment_pipe is not None:

            ADA_P = float(
                augment_pipe.p.cpu()
            )

            print(
                "ADA recuperado:",
                round(ADA_P, 4)
            )

    except Exception as e:

        print(
            "No fue posible recuperar "
            "la intensidad anterior de ADA."
        )

        print(
            "Se continuará con ADA desde 0."
        )

        ADA_P = 0.0


# ------------------------------------------------------------
# Nuevo entrenamiento
# ------------------------------------------------------------

else:

    RESUME = modelo_inicial

    print()
    print("Nuevo entrenamiento.")
    print(
        "Modelo inicial:",
        RESUME
    )


# ------------------------------------------------------------
# Construir comando
# ------------------------------------------------------------

cmd = [

    # -u = salida sin buffer.
    # Permite ver el entrenamiento en tiempo real.
    "python",
    "-u",

    # Lanzador optimizado para Colab
    "train_colab.py",

    # Mostrar estado cada 1 kimg
    "--tick=1",

    # Carpeta específica de este dataset
    f"--outdir={PROJECT_DIR}",

    # Configuración para GPU tipo T4
    "--cfg=11gb-gpu",

    # Dataset
    f"--data={DATASET}",

    # Adaptive Discriminator Augmentation
    "--aug=ada",
    "--target=0.6",

    # Reflexiones opcionales
    f"--mirror={mirror_horizontal}",
    f"--mirrory={mirror_vertical}",

    # No calcular FID durante el entrenamiento
    "--metrics=None",

    # Modelo inicial o último snapshot
    f"--resume={RESUME}",

    # Adecuado para Colab actual
    "--workers=2",

    # Con --tick=1:
    #
    # progreso = cada 1 kimg
    # snapshot = cada 4 ticks = 4 kimg
    #
    # En una T4 a 512 px:
    # progreso ≈ cada 4–5 minutos
    # snapshot ≈ cada 17–20 minutos
    "--snap=4",
]


# ------------------------------------------------------------
# Mantener contador e intensidad ADA al reanudar
# ------------------------------------------------------------

if START_KIMG > 0:

    cmd.append(
        f"--nkimg={START_KIMG}"
    )

    cmd.append(
        f"--initstrength={ADA_P}"
    )


# ------------------------------------------------------------
# Información para el usuario
# ------------------------------------------------------------

print()
print("========================================")
print("ENTRENAMIENTO")
print("========================================")

if START_KIMG > 0:

    print(
        f"Reanudando desde {START_KIMG} kimg."
    )

else:

    print(
        "Comenzando un nuevo modelo."
    )

print()
print(
    "Progreso en consola cada 1 kimg "
    "(aprox. 4–5 minutos en una T4 a 512 px)."
)

print(
    "Snapshots cada 4 kimg "
    "(aprox. 17–20 minutos)."
)

print()
print(
    "Puedes detener la celda cuando quieras."
)

print(
    "Al volver a ejecutarla se recuperará "
    "automáticamente el último snapshot guardado."
)

print("========================================")
print()


# ------------------------------------------------------------
# Ejecutar entrenamiento y mostrar salida en tiempo real
# ------------------------------------------------------------

process = subprocess.Popen(

    cmd,

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    bufsize=1,

    env={
        **os.environ,
        "PYTHONUNBUFFERED": "1"
    }
)


try:

    # Mostrar cada línea inmediatamente
    # en la consola de Colab
    for line in process.stdout:

        print(
            line,
            end="",
            flush=True
        )

    return_code = process.wait()

    if return_code != 0:

        raise subprocess.CalledProcessError(
            return_code,
            cmd
        )


except KeyboardInterrupt:

    print()
    print()
    print(
        "Entrenamiento detenido por el usuario."
    )

    process.terminate()
    process.wait()

    print(
        "Puedes volver a ejecutar esta celda "
        "para continuar desde el último "
        "snapshot guardado."
    )